# DNN Programming Assignment 2 — Problem 1: RNN, LSTM, and GRU on Sequences

**Student Name:** `Prahlad Vinayak Avadhani`  
**Student ID:** `2026AJ05253`  

---

## Documented Assumptions & Variant Configuration
- **Student ID:** `2026AJ05253` (last digit = `3`, odd).
- **Three Seeds:** `SEEDS = [3, 13, 23]` (derived from the last digit `3` of the Student ID).
- **Default Hidden Size (`h`):** `64` (matches Task 2 parameter derivation and Task 5b sweep midpoint).
- **Task 5(c) Individual Ablation (Odd ID):** LSTM forget-gate bias initialized to `0` vs. `1` at $T = 200$.
- **Training Budget:** Adam optimizer with `lr = 1e-3`, identical batch size and epoch budget across RNN, LSTM, and GRU within each comparison.

In [ ]:
import os
import time
import math
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ensure relative directories exist
os.makedirs('figures', exist_ok=True)
RESULTS_CSV_PATH = '../results.csv'

# Variant constants for Student ID 2026AJ05253 (last digit 3, odd)
STUDENT_ID = '2026AJ05253'
SEEDS = [3, 13, 23]
HIDDEN_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

def set_seed(seed: int):
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def log_result(problem, model, seed, seq_len='', test_acc='', macro_f1='',
               test_mse='', params='', sec_per_epoch='', val_loss_nats='',
               bits_per_char='', notes=''):
    """Append a single experiment run row to ../results.csv."""
    columns = [
        'problem', 'model', 'seed', 'seq_len', 'test_acc', 'macro_f1',
        'test_mse', 'params', 'sec_per_epoch', 'val_loss_nats', 'bits_per_char', 'notes'
    ]
    row = pd.DataFrame([{
        'problem': problem,
        'model': model,
        'seed': seed,
        'seq_len': seq_len,
        'test_acc': test_acc,
        'macro_f1': macro_f1,
        'test_mse': test_mse,
        'params': params,
        'sec_per_epoch': sec_per_epoch,
        'val_loss_nats': val_loss_nats,
        'bits_per_char': bits_per_char,
        'notes': notes
    }], columns=columns)
    write_header = not os.path.exists(RESULTS_CSV_PATH) or os.path.getsize(RESULTS_CSV_PATH) == 0
    row.to_csv(RESULTS_CSV_PATH, mode='a', header=write_header, index=False)

---
## Task 1 (P1.1): Data Pipelines (5 Points)

1. **UCI HAR Dataset:**
   - 9 inertial signal channels in fixed order: `body_acc_x, body_acc_y, body_acc_z, body_gyro_x, body_gyro_y, body_gyro_z, total_acc_x, total_acc_y, total_acc_z`.
   - Shapes: train `(7352, 128, 9)`, test `(2947, 128, 9)`, labels `1–6` (mapped to `0–5`).
   - Subject-wise validation split: hold out 4 of the 21 training subjects chosen with `np.random.default_rng(seed).choice(train_subjects, 4, replace=False)`.
   - Per-channel mean and std computed on the **training subset only**.
   - Class-balance plot saved to `figures/p1_har_class_balance.png`.
2. **Adding Problem Dataset:**
   - Uses the exact handout generator `make_adding(n, T, seed)` (`10,000` train with `seed=0`, `1,000` val with `seed=1`, `1,000` test with `seed=2`).
   - Unit test verifying two markers (one in each half) and target sum.

In [ ]:
# 1.1 UCI HAR Download and Raw Loader
HAR_DIR = 'data_har'
HAR_DATASET_FOLDER = os.path.join(HAR_DIR, 'UCI HAR Dataset')

CHANNELS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]
ACTIVITY_NAMES = ['Walking', 'Walking Upstairs', 'Walking Downstairs', 'Sitting', 'Standing', 'Laying']

def download_and_extract_har():
    if os.path.exists(HAR_DATASET_FOLDER):
        print('UCI HAR Dataset already extracted.')
        return
    os.makedirs(HAR_DIR, exist_ok=True)
    zip_path = os.path.join(HAR_DIR, 'har.zip')
    url = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
    print('Downloading UCI HAR dataset...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(HAR_DIR)
    inner_zip = os.path.join(HAR_DIR, 'UCI HAR Dataset.zip')
    if os.path.exists(inner_zip):
        with zipfile.ZipFile(inner_zip, 'r') as zf:
            zf.extractall(HAR_DIR)
    print('UCI HAR extraction complete.')

# TODO (Student): Implement load_har_raw() and get_har_splits(seed) with subject-wise validation holdout
# and training-only per-channel standardization, plus plot class balance.


In [ ]:
# 1.2 Adding Problem Generator (Provided in Assignment Specification)
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

# TODO (Student): Write unit test (assert cell) verifying that every sequence has
# exactly two markers (one in each half) and that y equals the sum of the two marked values.


---
## Task 2 (P1.2): RNN, LSTM, and GRU Cells from Scratch (10 Points)

- Implement `MyRNNCell`, `MyLSTMCell`, and `MyGRUCell` using only basic matrix/tensor operations (no `nn.RNN`, `nn.LSTM`, `nn.GRU`, or `nn.*Cell` inside).
- Copy weights from PyTorch's `nn.RNN`, `nn.LSTM`, and `nn.GRU` and verify outputs match within `1e-5`.
- Reconcile hand parameter counts for $d = 9, h = 64$ (`4,736`, `18,944`, `14,208`) with PyTorch's `sum(p.numel())` (`4,800`, `19,200`, `14,400`).

In [ ]:
class MyRNNCell(nn.Module):
    """
    Vanilla tanh RNN cell from scratch.
    Input shapes:
      x_t:    (B, d)
      h_prev: (B, h)
    Output shape:
      h_t:    (B, h)
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # TODO (Phase 2 Step 1): Define weight_ih, weight_hh, bias_ih, bias_hh
        pass

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor = None) -> torch.Tensor:
        # TODO (Phase 2 Step 1): Compute h_t = tanh(x_t @ W_ih^T + b_ih + h_prev @ W_hh^T + b_hh)
        pass


class MyLSTMCell(nn.Module):
    """
    LSTM cell from scratch matching PyTorch gate order [i, f, g, o].
    Input shapes:
      x_t:    (B, d)
      state:  tuple (h_prev, c_prev), each of shape (B, h)
    Output shape:
      (h_t, c_t): each of shape (B, h)
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # TODO (Phase 2 Step 2): Define weight_ih (4h, d), weight_hh (4h, h), bias_ih (4h,), bias_hh (4h,)
        pass

    def forward(self, x_t: torch.Tensor, state: tuple = None) -> tuple:
        # TODO (Phase 2 Step 2): Compute gates i, f, g, o, new cell state c_t, and new hidden state h_t
        pass


class MyGRUCell(nn.Module):
    """
    GRU cell from scratch matching PyTorch gate order [r, z, n].
    Input shapes:
      x_t:    (B, d)
      h_prev: (B, h)
    Output shape:
      h_t:    (B, h)
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # TODO (Phase 2 Step 3): Define weight_ih (3h, d), weight_hh (3h, h), bias_ih (3h,), bias_hh (3h,)
        pass

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor = None) -> torch.Tensor:
        # TODO (Phase 2 Step 3): Compute gates r, z, candidate n, and new hidden state h_t
        pass


In [ ]:
# Unit Tests & Parameter Count Verification for Task 2 (P1.2)
# TODO (Student): Copy weights from nn.RNN, nn.LSTM, nn.GRU, unroll over T steps,
# assert max absolute difference < 1e-5, and verify parameter counts for d=9, h=64.


---
## Task 3 (P1.3): UCI HAR Classification (10 Points)

- Train RNN, LSTM, and GRU classifiers reading 128 steps and classifying from the final hidden state $h_T$.
- Same hidden size (`h = 64`), Adam (`lr = 1e-3`), batch size, and epoch budget across all 3 models and 3 seeds (`[3, 13, 23]`).
- Report test accuracy (`mean ± std`), macro-F1 (`mean ± std`), parameter counts, and seconds/epoch.
- Save confusion matrix of the best model to `figures/p1_har_confusion_matrix.png`.

In [ ]:
# TODO (Student): Implement HARSequenceClassifier and run 3-seed comparison for RNN, LSTM, GRU.


---
## Task 4 (P1.4): Adding Problem (10 Points)

- Train RNN, LSTM, and GRU for $T \in \{50, 100, 200, 400\}$ across 3 seeds (`[3, 13, 23]`).
- Compare test MSE (`mean ± std`) against the `0.167` constant-prediction baseline.
- Plot learning curves at $T = 200$ (`figures/p1_adding_curves_T200.png`) and test MSE vs. $T$ (`figures/p1_adding_mse_vs_T.png`).
- Tabulate solved runs ($\text{test MSE} < 0.05$).

In [ ]:
# TODO (Student): Implement AddingRegressor and run experiments across T in {50, 100, 200, 400}.


---
## Task 5 (P1.5): Gradients, Ablations, and Analysis (15 Points)

- **5(a):** At $T = 100$, plot $\|\partial L / \partial h_t\|$ against $t$ on a log scale for RNN, LSTM, and GRU before and after training (`figures/p1_gradient_norms_T100.png`).
- **5(b):** Hidden-size sweep $h \in \{16, 32, 64, 128\}$ for LSTM and GRU on HAR (`figures/p1_hidden_size_sweep.png`).
- **5(c):** Odd Student ID (`2026AJ05253`) ablation: LSTM forget-gate bias initialized to `0` vs. `1` at $T = 200$ (`figures/p1_ablation_forget_bias.png`).
- **5(d):** One-page comparative analysis and wearable deployment recommendation.

In [ ]:
# TODO (Student): 5(a) Log-scale gradient norm ||dL/dh_t|| across time steps t=1..100 before and after training.


In [ ]:
# TODO (Student): 5(b) Hidden-size sweep {16, 32, 64, 128} for LSTM and GRU on HAR.


In [ ]:
# TODO (Student): 5(c) Individual ablation (Odd Student ID): LSTM forget-gate bias 0 vs 1 at T=200.


### Task 5(d): Comparative Analysis & Wearable Recommendation

*(To be completed by student after viewing experimental tables and plots.)*